# High-z Example 10: Redshift Distribution

**EPS Research — High-z Kinematic Corpus Z1**

31 galaxies spanning z = 4.26-5.68.
9 galaxies lie at z > 5, approaching cosmic reionization.
Maximum redshift: DC773957 at z = 5.6773.

**Corpus:** Flynn (2026), Zenodo DOI: 10.5281/zenodo.21834678  
**arXiv:** 2605.25339  
**Source:** Jones et al. (2021), MNRAS 507, 3540; Le Fevre et al. (2020)  
**Dependencies:** Python 3, numpy, matplotlib

In [ ]:
# ── Colab setup: auto-download corpus from Zenodo ─────────────
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    import urllib.request
    CORPORA = {
        'high_z_kinematic_corpus_Z1.json': 'https://zenodo.org/records/21834678/files/high_z_kinematic_corpus_Z1.json',
    }
    for filename, url in CORPORA.items():
        if not os.path.exists(filename):
            print(f"Downloading {filename}...")
            urllib.request.urlretrieve(url, filename)
            print(f"  ✓ {filename}")
        else:
            print(f"  Already present: {filename}")
    if not os.path.exists("high_z_kinematic_corpus_Z1_flat.csv"):
        import json, csv
        _g = json.load(open("high_z_kinematic_corpus_Z1.json"))["galaxies"]
        _k = [k for k in _g[0] if not isinstance(_g[0][k], (dict, list))]
        with open("high_z_kinematic_corpus_Z1_flat.csv", "w", newline="") as _f:
            _w = csv.DictWriter(_f, fieldnames=_k)
            _w.writeheader()
            for _r in _g:
                _w.writerow({k: ("" if _r.get(k) is None else _r.get(k)) for k in _k})
        print("  built high_z_kinematic_corpus_Z1_flat.csv from the JSON (the Zenodo record has no flat CSV)")
    print("Ready.")
else:
    print("Running locally — corpus files loaded from working directory.")


In [ ]:
import csv
import numpy as np
import matplotlib.pyplot as plt

rows = []
with open('high_z_kinematic_corpus_Z1_flat.csv') as f:
    for r in csv.DictReader(f):
        rows.append(r)
print(f"Total galaxies: {len(rows)}")


In [ ]:
import numpy as np
zs = [float(r['redshift']) for r in rows]
classes = [r['class_jones2021'] for r in rows]

print(f"Redshift range: {min(zs):.4f} -- {max(zs):.4f}")
print(f"Median: {np.median(zs):.4f}")
print(f"z > 5: {sum(1 for z in zs if z > 5)} galaxies")
print(f"\nMax redshift galaxy:")
max_idx = zs.index(max(zs))
print(f"  {rows[max_idx]['galaxy']} at z = {max(zs):.4f}")

COLORS = {'ROT':'#2ecc71','MER':'#e74c3c','DIS':'#f39c12','UNC':'#95a5a6'}
bins = [4.2, 4.4, 4.6, 4.8, 5.0, 5.2, 5.4, 5.6, 5.8]

fig, ax = plt.subplots(figsize=(9, 4))
bottom = np.zeros(len(bins)-1)
for cls, color in COLORS.items():
    vals = [float(r['redshift']) for r in rows if r['class_jones2021']==cls]
    counts, _ = np.histogram(vals, bins=bins)
    ax.bar(bins[:-1], counts, width=0.18, bottom=bottom,
           color=color, label=f'{cls} ({len(vals)})', alpha=0.85, align='edge')
    bottom += counts
ax.axvline(5.0, color='black', ls='--', lw=1.5, label='z=5')
ax.set_xlabel('Redshift z', fontsize=12)
ax.set_ylabel('N galaxies', fontsize=12)
ax.set_title('Redshift Distribution — Z1 Corpus (N=31)\n'
             'Stacked by morpho-kinematic class', fontsize=11)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('hz10_redshift_distribution.png', dpi=150, bbox_inches='tight')
plt.show()